---
# Clinical Data Quality Engine (DQIE)
# Notebook 04 — Reconciliation
# Purpose: Run the full reconciliation pipeline (Module 5)
---

# Preparations
---

## Setup do ambiente

In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print("PYTHONPATH OK:", PROJECT_ROOT)

## Imports of reconciliation engine

In [ ]:
from src._5_reconciliation.reconciliation_engine import run_reconciliation_pipeline

# Load Silver Layer
---

In [ ]:
df_patients = pd.read_parquet("../data/_2_silver/patients.parquet")
df_injuries = pd.read_parquet("../data/_2_silver/injuries.parquet")
df_sessions = pd.read_parquet("../data/_2_silver/sessions.parquet")
df_clinical = pd.read_parquet("../data/_2_silver/clinical_reports.parquet")
df_ocr_json = pd.read_parquet("../data/_2_silver/ocr_extracted.parquet")
df_ocr_images = pd.read_parquet("../data/_2_silver/ocr_images.parquet")

print("Silver Layer loaded.")

# Run Reconciliation Pipeline
---

In [ ]:
print("\n## Running full reconciliation pipeline...")

reconciliation_output = run_reconciliation_pipeline(
    patients=df_patients,
    injuries=df_injuries,
    sessions=df_sessions,
    csv_data=df_sessions,
    sql_data=df_sessions,
    ocr_text=df_ocr_json,
    clinical_json=df_clinical,
    ocr_images=df_ocr_images
)

print("\nPipeline completed.")

# Display Results
---

In [ ]:
for key, value in reconciliation_output.items():
    print(f"\n### {key} ###")
    display(value)

# Summary
---

In [ ]:
print("\n--- RECONCILIATION SUMMARY ---")
print(f"Value anomalies: {len(reconciliation_output['value_anomalies'])}")
print(f"Relational anomalies: {len(reconciliation_output['relational_anomalies'])}")
print(f"Temporal anomalies: {len(reconciliation_output['temporal_anomalies'])}")
print(f"Source anomalies: {len(reconciliation_output['source_anomalies'])}")
print(f"Anomaly summary rows: {len(reconciliation_output['anomalies_summary'])}")
print(f"Entity reconciliation rows: {len(reconciliation_output['entities_reconciliation'])}")

print("\nDataset reconciled. Proceed to Notebook 05 — DQI Scoring.")

# Save reconciliation outputs to Gold Layer

In [ ]:
import json

# Save reconciliation outputs to Gold Layer
output_dir = "../data/_3_gold/reconciliation/"
os.makedirs(output_dir, exist_ok=True)

# anomalies_summary salva direto
reconciliation_output["anomalies_summary"].to_parquet(
    f"{output_dir}/anomalies_summary.parquet", index=False
)

# entities_reconciliation precisa converter colunas problemáticas
entities_reconciliation = reconciliation_output["entities_reconciliation"].copy()

# Convert dict columns to JSON strings
if "source_values" in entities_reconciliation.columns:
    entities_reconciliation["source_values"] = (
        entities_reconciliation["source_values"]
        .apply(lambda x: json.dumps(x) if isinstance(x, dict) else str(x))
    )

# Convert chosen_value to string
entities_reconciliation["chosen_value"] = (
    entities_reconciliation["chosen_value"]
    .apply(lambda x: json.dumps(x) if isinstance(x, dict) else str(x))
)

# Convert all object columns to string
for col in entities_reconciliation.columns:
    if entities_reconciliation[col].dtype == "object":
        entities_reconciliation[col] = entities_reconciliation[col].astype("string")

entities_reconciliation.to_parquet(
    f"{output_dir}/entities_reconciliation.parquet",
    index=False
)

print("Reconciliation outputs saved to Gold Layer.")
